In [55]:
import pandas as pd
import numpy as np
import gffutils
import tqdm
from thefuzz import process

In [56]:
gene_df = pd.read_csv('data/GM12878_K562_18377_gene_expr_fromXpresso.csv')
transcript_df = pd.read_csv('data/K562_GM12878_transcript_tpm.txt', index_col=0, sep='\t')
se_events_df = pd.read_csv('data/transcript_SE_f1.psi', sep='\t', index_col=0)
gene_ids = gene_df['ENSID'].values
transcript_ids = transcript_df.index.values
gene_ids

array(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', ...,
       'ENSG00000259511', 'ENSG00000259571', 'ENSG00000259680'],
      dtype=object)

In [57]:
transcript_ids = np.array([x.split('.')[0] for x in transcript_ids])
transcript_ids

array(['ENST00000373020', 'ENST00000494424', 'ENST00000496771', ...,
       'ENST00000648949', 'ENST00000650266', 'ENST00000648650'],
      dtype='<U15')

In [58]:
se_events_v = np.array([x for x in se_events_df.index.values])
se_events = []
for i in se_events_v:
    event = i.split(';')
    gene = event[0].split('.')[0]
    new_event = ";".join([gene] + event[1:])
    se_events.append(new_event)
se_events[:10]

['ENSG00000000419;SE:chr20:50940933-50941129:50941209-50942031:-',
 'ENSG00000000457;SE:chr1:169854964-169855796:169855957-169859041:-',
 'ENSG00000000460;SE:chr1:169798958-169800883:169800971-169802621:+',
 'ENSG00000000460;SE:chr1:169806088-169807791:169807929-169821679:+',
 'ENSG00000000971;SE:chr1:196676065-196677476:196677667-196679623:+',
 'ENSG00000001084;SE:chr6:53514497-53516109:53516222-53520778:-',
 'ENSG00000001460;SE:chr1:24391679-24391924:24392067-24401319:-',
 'ENSG00000001461;SE:chr1:24419640-24433104:24433213-24440172:+',
 'ENSG00000001461;SE:chr1:24456273-24457734:24457833-24458888:+',
 'ENSG00000001497;SE:chrX:65524262-65524564:65524614-65524965:-']

In [59]:
def infer_intron(upstream, exon):
    intron_start = upstream.end
    intron_end = exon.start
    if intron_start > intron_end:
        raise ValueError(f"Intron start {intron_start} is greater than intron end {intron_end}.")
    return (intron_start, intron_end)

In [60]:
# load gtf file
db = gffutils.FeatureDB('data/db.gtf.db', keep_order=True)

In [ ]:
# event definition looks like this:
# ENSG00000228794;SE:chr1:829104-847654:847806-851927:+
transcript_ids = set(transcript_ids)
gene_ids = set(gene_ids)
artifical_events = {}

for gene in tqdm.tqdm(db.features_of_type('gene')):
    gene_id = gene.id.split('.')[0]  # Get the gene ID without version
    if gene_id not in gene_ids:
        continue

    for transcript in db.children(gene, featuretype='transcript', order_by='start'):
        exons = list(db.children(transcript, featuretype='exon', order_by='start'))
        if len(exons) < 3:
            continue # We need at least 3 exons to define an event

        strand = transcript.strand
        chrom = transcript.chrom
        t_id = transcript.id.split('.')[0]  # Get the transcript ID without version
        if t_id not in transcript_ids:
            continue
        # Go through internal exons only
        for i in range(1, len(exons) - 1):
            upstream = exons[i - 1]
            skipped = exons[i]
            downstream = exons[i + 1]
            upstream_intron = infer_intron(upstream, skipped)
            downstream_intron = infer_intron(skipped, downstream)
            

            event = f"{gene_id};AR:{chrom}:{upstream_intron[0]}-{upstream_intron[1]}:{downstream_intron[0]}-{downstream_intron[1]}:{strand}"
            if event in artifical_events:
                artifical_events[event].append(t_id)
                continue
            artifical_events[event] = [t_id]
print(f"Found {len(artifical_events)} artificial events")

23796it [00:44, 537.07it/s] 

Found 208980 unique artificial events


In [75]:
se_event = se_events[0]
se_event

'ENSG00000000419;SE:chr20:50940933-50941129:50941209-50942031:-'

In [77]:
process.extract(se_event, artifical_events.keys(), scorer=process.fuzz.ratio, limit=5)

[('ENSG00000000419;AR:chr20:50940933-50941129:50941209-50942031:-', 97),
 ('ENSG00000000419;AR:chr20:50940933-50942031:50942126-50945737:-', 83),
 ('ENSG00000000419;AR:chr20:50941209-50942031:50942126-50945737:-', 80),
 ('ENSG00000000419;AR:chr20:50936262-50940865:50940933-50942031:-', 80),
 ('ENSG00000000419;AR:chr20:50936262-50940865:50940955-50942031:-', 80)]

In [78]:
def cut_event_name(event):
    """
    Cut the event name to get the gene ID and event type.
    """
    parts = event.split(';')
    gene_id = parts[0]
    event_info = parts[1].split(':')
    assembled_info = ":".join(event_info[1:])
    return f"{gene_id};{assembled_info}"

cut_event_name(se_event)

'ENSG00000000419;chr20:50940933-50941129:50941209-50942031:-'

In [79]:
se_events_set = set([cut_event_name(x) for x in se_events])

filtered_ar_events = {}
for ar in artifical_events:
    ar_cut = cut_event_name(ar)
    if ar_cut not in se_events_set:
        filtered_ar_events[ar] = artifical_events[ar]

print(f"Filtered {len(artifical_events) - len(filtered_ar_events)} artificial events that are also SE events.\nRemaining: {len(filtered_ar_events)}")

Filtered 13973 artificial events that are also SE events.
Remaining: 195007


In [80]:
filtered_ar_events

{'ENSG00000000003;AR:chrX:100629986-100630759:100630866-100632485:-': ['ENST00000373020'],
 'ENSG00000000003;AR:chrX:100630866-100632485:100632568-100633405:-': ['ENST00000373020'],
 'ENSG00000000003;AR:chrX:100632568-100633405:100633539-100633931:-': ['ENST00000373020'],
 'ENSG00000000003;AR:chrX:100633539-100633931:100634029-100635178:-': ['ENST00000373020',
  'ENST00000494424'],
 'ENSG00000000003;AR:chrX:100634029-100635178:100635252-100635558:-': ['ENST00000373020',
  'ENST00000494424'],
 'ENSG00000000003;AR:chrX:100635252-100635558:100635746-100636608:-': ['ENST00000373020'],
 'ENSG00000000003;AR:chrX:100635252-100635558:100635746-100636793:-': ['ENST00000494424'],
 'ENSG00000000003;AR:chrX:100635746-100636793:100637104-100639945:-': ['ENST00000494424'],
 'ENSG00000000005;AR:chrX:100585066-100585231:100585362-100593895:+': ['ENST00000373031'],
 'ENSG00000000005;AR:chrX:100585362-100593895:100594035-100594261:+': ['ENST00000373031'],
 'ENSG00000000005;AR:chrX:100594035-100594261:10